# Milestone 3 — Auto-Label Audit, Partition Split, Train EDA & 6-Table HF Packaging

Run on a **Google Colab CPU** runtime. Each step is guarded so you can safely run only what is missing on Drive:

1. **Setup & verified restore:** mounts Drive, checks out `main`, verifies `m2-full-no-empty-01` identity, and copies only what is needed.
2. **Optional auto-label & blind audit:** run only if `labels/auto/` is not yet generated on Drive (`RUN_AUTO_LABEL = False` by default).
3. **Freeze `partitions.csv`:** creates the deterministic per-range split (4 manual + 4 auto in Test, 4 manual + 4 auto in Valid, remaining labeled papers in Train) and verifies it.
4. **Train-only EDA (`analysis.py --partitions`):** computes feature justification statistics strictly on Train papers (no Valid/Test leakage).
5. **Package 6 Parquet tables (`package_dataset.py`):** streams filtered M2 shards with bounded RAM, combines with M1 (`papers`, `references`) and `predictions` into `<owner>/data/<table>/*.parquet`.
6. **Verify & upload to Hugging Face (`hf_dataset.py --strict-schema`):** dry-run first, then public upload using Colab Secret `HF_TOKEN` and streaming reload check.

In [ ]:
M1_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1"
M2_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1/m2"
M2_SOURCE_RUN_ID = "m2-full-01"
M2_DERIVED_ID = "m2-full-no-empty-01"
M3_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1/m3"
CODE_REF = "main"
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"
LOCAL = "/content/m3-pack-local"
MY_RANGE = "thien"  # thien / tien / hieu / dang
HF_USER = "<your-hf-username>"

## 1. Mount Drive, checkout code, and inspect M3 Drive state

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
repo_dir = Path("/content/CodeRepo-m3-pack")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", CODE_REF], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
print("code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pyarrow",
        "pandas",
        "tqdm",
        "huggingface_hub",
        "datasets",
    ],
    check=True,
)

M2_SOURCE_RUN = Path(M2_DRIVE_DIR) / M2_SOURCE_RUN_ID
M2_RUN = Path(M2_DRIVE_DIR) / "derived" / M2_DERIVED_ID
M3 = Path(M3_DRIVE_DIR)
LABELS = M3 / "labels"
SELECTION = M3 / "selection.csv"
PARTITIONS = M3 / "partitions.csv"
M1_BY_RANGE = M3 / "m1_tables_by_range"
REFS_PARQUET = M3 / "m1_tables_all" / "data" / "references"
M2_OUT_LOCAL = Path(LOCAL) / M2_DERIVED_ID / "out"
CORPUS_LOCAL = Path(LOCAL) / "corpus"
PACKAGES_DIR = M3 / "packages"

identity = json.loads((M2_RUN / "identity.json").read_text(encoding="utf-8"))
manifest = (M2_SOURCE_RUN / "M2_MANIFEST.json").read_bytes()
if (
    identity["run_id"] != M2_SOURCE_RUN_ID
    or identity["manifest_sha256"] != hashlib.sha256(manifest).hexdigest()
):
    raise ValueError("Derived M2 output does not match M2_MANIFEST.json")

n_manual = len(list((LABELS / "manual").glob("*.csv"))) if (LABELS / "manual").is_dir() else 0
n_auto = len(list((LABELS / "auto").glob("*.csv"))) if (LABELS / "auto").is_dir() else 0
print(f"manual CSVs: {n_manual}, auto CSVs: {n_auto}, partitions exists: {PARTITIONS.exists()}")

## 2. Optional: restore BibTeX / references and run auto-label + blind audit

Skip this cell if `labels/auto/` already has >= 2,000 CSVs on Drive. Toggle `RUN_AUTO_LABEL = True` only if you need to generate `labels/auto/`.

In [ ]:
from tqdm.auto import tqdm

from src.m3 import auto_label, crosscheck
from src.m3.restore_inputs import restore_bibtex, restore_references

RUN_AUTO_LABEL = False
RUN_AUTO_EVAL = False
RUN_AUTO_CROSSCHECK_SAMPLE = False


def progress(bar):
    def update(done, total):
        bar.total = total
        bar.update(done - bar.n)

    return update


if RUN_AUTO_LABEL or RUN_AUTO_EVAL or RUN_AUTO_CROSSCHECK_SAMPLE:
    with tqdm(desc="bibtex shards", unit="shard") as bar:
        print(restore_bibtex(M2_RUN, M2_OUT_LOCAL, on_progress=progress(bar)))

if RUN_AUTO_LABEL:
    args = [
        "run",
        "--m2-dir",
        str(M2_OUT_LOCAL),
        "--refs-dir",
        str(REFS_PARQUET),
        "--labels-dir",
        str(LABELS),
        "--total-corpus-papers",
        "20000",
        "--ratio",
        "0.10",
        "--min-papers",
        "2000",
        "--seed",
        "20261117",
    ]
    if SELECTION.exists():
        args += ["--selection-csv", str(SELECTION)]
    auto_label.main(args)

if RUN_AUTO_EVAL:
    auto_label.main(
        [
            "evaluate",
            "--m2-dir",
            str(M2_OUT_LOCAL),
            "--refs-dir",
            str(REFS_PARQUET),
            "--labels-dir",
            str(LABELS),
            "--out-dir",
            str(M3 / "auto_eval"),
        ]
    )

if RUN_AUTO_CROSSCHECK_SAMPLE:
    with tqdm(desc="references.json", unit="file") as bar:
        print(
            restore_references(M2_SOURCE_RUN, M1_DRIVE_DIR, CORPUS_LOCAL, on_progress=progress(bar))
        )
    crosscheck.main(
        [
            "sample",
            "--method",
            "auto",
            "--m2-dir",
            str(M2_OUT_LOCAL),
            "--corpus-dir",
            str(CORPUS_LOCAL),
            "--labels-dir",
            str(LABELS),
            "--selection-csv",
            str(SELECTION),
            "--sheet",
            str(M3 / "auto_crosscheck_sheet.csv"),
            "--key",
            str(M3 / "auto_crosscheck_key.csv"),
        ]
    )

## 3. Create & verify frozen `partitions.csv` (4+4 Test, 4+4 Valid, remaining Train)

In [ ]:
from src.m3 import split_partitions

if not PARTITIONS.exists():
    args = [
        "create",
        "--labels-dir",
        str(LABELS),
        "--out",
        str(PARTITIONS),
        "--seed",
        "20261116",
    ]
    if SELECTION.exists():
        args += ["--selection-csv", str(SELECTION)]
    split_partitions.main(args)
else:
    print("partitions.csv already exists — running verification only.")

verify_args = ["verify", "--labels-dir", str(LABELS), "--partitions", str(PARTITIONS)]
if SELECTION.exists():
    verify_args += ["--selection-csv", str(SELECTION)]
split_partitions.main(verify_args)

## 4. Run Train-only EDA (`analysis.py --partitions`) for feature justification & poster

In [ ]:
from tqdm.auto import tqdm

from src.m3 import analysis
from src.m3.restore_inputs import restore_bibtex, restore_references

RUN_TRAIN_EDA = False
if RUN_TRAIN_EDA:

    def _prog(bar):
        def update(done, total):
            bar.total = total
            bar.update(done - bar.n)

        return update

    with tqdm(desc="bibtex shards", unit="shard") as bar:
        restore_bibtex(M2_RUN, M2_OUT_LOCAL, on_progress=_prog(bar))
    with tqdm(desc="references.json", unit="file") as bar:
        restore_references(M2_SOURCE_RUN, M1_DRIVE_DIR, CORPUS_LOCAL, on_progress=_prog(bar))
    analysis.main(
        [
            "--m2-dir",
            str(M2_OUT_LOCAL),
            "--corpus-dir",
            str(CORPUS_LOCAL),
            "--labels-dir",
            str(LABELS),
            "--partitions",
            str(PARTITIONS),
            "--out-dir",
            str(M3 / "analysis_train"),
        ]
    )
else:
    print("Set RUN_TRAIN_EDA = True once partitions.csv is ready.")

## 5. Package 6 Parquet tables (`by-range` or `combined`, bounded RAM)

- Leave `ALLOW_MISSING_PREDICTIONS = True` to pre-package the 5 M1+M2 tables now.
- Once `predictions_by_range/` is ready, set `ALLOW_MISSING_PREDICTIONS = False` and `SKIP_M2 = True` to attach `predictions`, run `--strict-schema`, and write dataset cards in seconds.

In [ ]:
from src.m3 import package_dataset

ALLOW_MISSING_PREDICTIONS = True
SKIP_M2 = False
MODE = "by-range"  # or "combined"
PREDICTIONS_DIR = M3 / "predictions_by_range"

pack_args = [
    "--m1-dir",
    str(M1_BY_RANGE),
    "--m2-dir",
    str(M2_RUN / "out"),
    "--labels-dir",
    str(LABELS),
    "--out-dir",
    str(PACKAGES_DIR),
    "--mode",
    MODE,
]
if PREDICTIONS_DIR.exists():
    pack_args += ["--predictions-dir", str(PREDICTIONS_DIR)]
if ALLOW_MISSING_PREDICTIONS:
    pack_args.append("--allow-missing-predictions")
if SKIP_M2:
    pack_args.append("--skip-m2")
package_dataset.main(pack_args)

## 6. Validate & upload Hugging Face dataset for `MY_RANGE`

Run with `DRY_RUN = True` first to verify `--strict-schema` and the YAML dataset card (`data/<table>/*.parquet`), then set `DRY_RUN = False` to publish (requires Colab Secret `HF_TOKEN`).

In [ ]:
from google.colab import userdata

from src.m3 import hf_dataset

DRY_RUN = True
HF_REPO_ID = f"{HF_USER}/arxiv-nhom1-{MY_RANGE}"
PKG = PACKAGES_DIR / MY_RANGE

hf_args = [
    "--package-dir",
    str(PKG),
    "--repo-id",
    HF_REPO_ID,
    "--range",
    MY_RANGE,
    "--labels-dir",
    str(LABELS),
    "--strict-schema",
]
if DRY_RUN:
    hf_args.append("--dry-run")
else:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
hf_dataset.main(hf_args)